In [1]:
# Chia tap du lieu theo thoi gian (2020-2025)
# - From: HDFS `/user/data/raw`
# - To (copy): `/user/data/train`, `/user/data/val`, `/user/data/test`
# - 70% train, 10% val, 20% test

In [2]:
import os
import re
import shutil
import subprocess
from typing import List, Tuple

RAW_DIR = "/user/data/raw"
TRAIN_DIR = "/user/data/train"
VAL_DIR = "/user/data/val"
TEST_DIR = "/user/data/test"

# Match YYYY-MM in filename, keep 2020-2025
MONTH_RE = re.compile(r"(20(2[0-5]))-(0[1-9]|1[0-2])")

def resolve_hdfs_cmd() -> str:
    env_cmd = os.environ.get("HDFS_CMD")
    if env_cmd and os.path.exists(env_cmd):
        return env_cmd
    hdfs = shutil.which("hdfs")
    if hdfs:
        return hdfs
    hadoop_home = os.environ.get("HADOOP_HOME")
    if hadoop_home:
        candidate = os.path.join(hadoop_home, "bin", "hdfs")
        if os.path.exists(candidate):
            return candidate
    home = os.path.expanduser("~")
    candidates = [
        os.path.join(home, "hadoop-3.3.6", "bin", "hdfs"),
        "/usr/local/hadoop/bin/hdfs",
        "/usr/lib/hadoop/bin/hdfs",
    ]
    for candidate in candidates:
        if os.path.exists(candidate):
            return candidate
    raise RuntimeError(
        "Khong tim thay lenh 'hdfs'. Hay set HADOOP_HOME hoac HDFS_CMD truoc khi chay."
    )

HDFS_CMD = resolve_hdfs_cmd()

def run(args: List[str]) -> str:
    result = subprocess.run(args, text=True, capture_output=True)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed: {' '.join(args)}\n{result.stderr}")
    return result.stdout

def hdfs_exists(path: str) -> bool:
    result = subprocess.run([HDFS_CMD, "dfs", "-test", "-e", path])
    return result.returncode == 0

def list_hdfs_files(hdfs_dir: str) -> List[str]:
    out = run([HDFS_CMD, "dfs", "-ls", hdfs_dir])
    paths = []
    for line in out.splitlines():
        parts = line.split()
        if len(parts) >= 8 and parts[0][0] in "-d":
            paths.append(parts[-1])
    return paths

def extract_year_month(path: str) -> Tuple[int, int] | None:
    m = MONTH_RE.search(os.path.basename(path))
    if not m:
        return None
    year = int(m.group(1)[:4])
    month = int(m.group(3))
    return year, month

raw_paths = list_hdfs_files(RAW_DIR)
candidates = []
for p in raw_paths:
    ym = extract_year_month(p)
    if ym is None:
        continue
    year, month = ym
    if 2020 <= year <= 2025:
        candidates.append((year, month, p))

if not candidates:
    raise ValueError("Khong tim thay file parquet 2020-2025 trong /user/data/raw")

# Sort by time (year, month)
candidates.sort(key=lambda x: (x[0], x[1]))
files = [p for _, _, p in candidates]
total = len(files)

train_count = int(total * 0.7)
val_count = max(1, int(total * 0.1))
test_count = total - train_count - val_count
if train_count <= 0 or val_count <= 0 or test_count <= 0:
    train_count = max(1, total - 2)
    val_count = 1
    test_count = total - train_count - val_count

train_files = files[:train_count]
val_files = files[train_count:train_count + val_count]
test_files = files[train_count + val_count:]

print(f"Tong file: {total}")
print(f"Train: {len(train_files)}, Val: {len(val_files)}, Test: {len(test_files)}")
print("First/last in train:", os.path.basename(train_files[0]), os.path.basename(train_files[-1]))
print("First/last in val:", os.path.basename(val_files[0]), os.path.basename(val_files[-1]))
print("First/last in test:", os.path.basename(test_files[0]), os.path.basename(test_files[-1]))

# Create destination dirs
run([HDFS_CMD, "dfs", "-mkdir", "-p", TRAIN_DIR, VAL_DIR, TEST_DIR])

def copy_files(src_list: List[str], dest_dir: str) -> None:
    for src in src_list:
        name = os.path.basename(src)
        dest = f"{dest_dir}/{name}"
        if hdfs_exists(dest):
            print(f"Skip (exists): {dest}")
            continue
        run([HDFS_CMD, "dfs", "-cp", src, dest])
        print(f"Copied: {src} -> {dest}")

copy_files(train_files, TRAIN_DIR)
copy_files(val_files, VAL_DIR)
copy_files(test_files, TEST_DIR)

print("Done.")

Tong file: 72
Train: 50, Val: 7, Test: 15
First/last in train: yellow_tripdata_2020-01.parquet yellow_tripdata_2024-02.parquet
First/last in val: yellow_tripdata_2024-03.parquet yellow_tripdata_2024-09.parquet
First/last in test: yellow_tripdata_2024-10.parquet yellow_tripdata_2025-12.parquet
Copied: /user/data/raw/yellow_tripdata_2020-01.parquet -> /user/data/train/yellow_tripdata_2020-01.parquet
Copied: /user/data/raw/yellow_tripdata_2020-02.parquet -> /user/data/train/yellow_tripdata_2020-02.parquet
Copied: /user/data/raw/yellow_tripdata_2020-03.parquet -> /user/data/train/yellow_tripdata_2020-03.parquet
Copied: /user/data/raw/yellow_tripdata_2020-04.parquet -> /user/data/train/yellow_tripdata_2020-04.parquet
Copied: /user/data/raw/yellow_tripdata_2020-05.parquet -> /user/data/train/yellow_tripdata_2020-05.parquet
Copied: /user/data/raw/yellow_tripdata_2020-06.parquet -> /user/data/train/yellow_tripdata_2020-06.parquet
Copied: /user/data/raw/yellow_tripdata_2020-07.parquet -> /user/